In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import lpips
from tqdm import tqdm
from torch.utils.data import DataLoader

from utils import (
    SinogramNoise,
    calculate_mse,
    calculate_psnr,
    calculate_ssim,
    calculate_correlation,
    calculate_gmsd,
    calculate_lpips,
)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
from dival import get_standard_dataset

dataset = get_standard_dataset(
    "custom",
    data_path="../data/ct_ellipses_dataset",
    sinogram_shape=(256, 183),
    image_shape=(128, 128),
    parts_len={"train": 80, "validation": 10, "test": 10},
    impl="skimage",
)

transform = SinogramNoise(mean=0.0, std=0.0, p=1.0)
test_dataset = dataset.create_torch_dataset(part="test", transform=transform)

batch_size = 1
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [ ]:
from models.Pix2Pix_128_V2 import UnetGenerator, UnetGeneratorSmall
from models.conv_netx import ConvNetX_128

MODEL_CONFIGS = [
    {"name": "UNetGeneratorSmall", "path": "../models/model1.pth", "class": UnetGeneratorSmall},
    {"name": "ConvNetX", "path": "../models/model2.pth", "class": ConvNetX_128},
    {"name": "Pix2Pix", "path": "../models/model3.pth", "class": UnetGenerator},
]

models = []
for cfg in MODEL_CONFIGS:
    model = cfg["class"]().to(device)
    checkpoint = torch.load(cfg["path"], map_location=device)
    if isinstance(checkpoint, dict) and "generator_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["generator_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    models.append(model)
    print(f"Loaded: {cfg['name']} from {cfg['path']}")

In [ ]:
lpips_net = lpips.LPIPS(net="vgg", verbose=False).to(device)

metric_names = ["MSE", "SSIM", "PSNR", "Correlation", "GMSD", "LPIPS"]
metric_fns = {
    "MSE": lambda gt, pred: calculate_mse(gt, pred),
    "SSIM": lambda gt, pred: calculate_ssim(gt, pred),
    "PSNR": lambda gt, pred: calculate_psnr(gt, pred, max_val=1.0),
    "Correlation": lambda gt, pred: calculate_correlation(gt, pred),
    "GMSD": lambda gt, pred: calculate_gmsd(gt, pred),
    "LPIPS": lambda gt, pred: calculate_lpips(gt, pred, lpips_net),
}

# Per-model, per-metric lists of per-image scores
results = {cfg["name"]: {m: [] for m in metric_names} for cfg in MODEL_CONFIGS}

with torch.no_grad():
    for sino, img in tqdm(test_loader, desc="[Evaluating]", leave=False):
        sino = sino.unsqueeze(1).to(device, non_blocking=True)
        img = img.unsqueeze(1).to(device, non_blocking=True)

        for i, cfg in enumerate(MODEL_CONFIGS):
            output = models[i](sino)
            for m_name, m_fn in metric_fns.items():
                results[cfg["name"]][m_name].append(m_fn(img, output))

In [ ]:
# Summary report
model_names = [cfg["name"] for cfg in MODEL_CONFIGS]

header = f"{'Metric':<14}" + "".join(f"{name:<16}" for name in model_names)
print(header)
print("-" * len(header))

for m in metric_names:
    row = f"{m:<14}"
    for name in model_names:
        vals = results[name][m]
        avg = sum(vals) / len(vals)
        row += f"{avg:<16.6f}"
    print(row)

### Rozkłady metryk

In [ ]:
# Distribution plots — one figure per metric, 3 overlaid KDE curves
from scipy.stats import gaussian_kde

colors = ["#4C72B0", "#DD8452", "#55A868"]
model_names = [cfg["name"] for cfg in MODEL_CONFIGS]

for m in metric_names:
    plt.figure(figsize=(10, 5))
    for i, name in enumerate(model_names):
        vals = np.array(results[name][m])
        if len(vals) > 1 and np.std(vals) > 0:
            kde = gaussian_kde(vals)
            x = np.linspace(vals.min(), vals.max(), 300)
            plt.plot(x, kde(x), color=colors[i], linewidth=2, label=name)
            plt.fill_between(x, kde(x), alpha=0.2, color=colors[i])
        else:
            plt.axvline(vals[0], color=colors[i], linewidth=2, label=name)
    plt.title(f"Rozkład {m}")
    plt.xlabel(m)
    plt.ylabel("Gęstość")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Combined distribution plot
from scipy.stats import gaussian_kde

selected_metrics = ["MSE", "SSIM", "PSNR", "LPIPS"]
colors = ["#4C72B0", "#DD8452", "#55A868"]
model_names = [cfg["name"] for cfg in MODEL_CONFIGS]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, m in zip(axes, selected_metrics):
    for i, name in enumerate(model_names):
        vals = np.array(results[name][m])
        if len(vals) > 1 and np.std(vals) > 0:
            kde = gaussian_kde(vals)
            x = np.linspace(vals.min(), vals.max(), 300)
            ax.plot(x, kde(x), color=colors[i], linewidth=2, label=name)
            ax.fill_between(x, kde(x), alpha=0.2, color=colors[i])
        else:
            ax.axvline(vals[0], color=colors[i], linewidth=2, label=name)
    ax.set_title(f"{m}")
    ax.set_xlabel(m)
    ax.set_ylabel("Gęstość")
    ax.legend()

fig.tight_layout()
plt.show()

### Wizualna ocena jakości rekonstrukcji

In [ ]:
# Reconstruction comparison: 3 random images × 5 methods (GT, FBP, 3 models)
import random
import odl
from dival.reconstructors.odl_reconstructors import FBPReconstructor

# FBP reconstructor
reco_space = odl.uniform_discr(
    min_pt=[-0.13, -0.13], max_pt=[0.13, 0.13],
    shape=(128, 128), dtype=np.float32,
)
reco_geometry = odl.tomo.parallel_beam_geometry(reco_space, num_angles=256)
reco_ray_trafo = odl.tomo.RayTransform(reco_space, reco_geometry, impl="skimage")
fbp_reconstructor = FBPReconstructor(reco_ray_trafo)

test_pairs = dataset.get_data_pairs(part="test")

# Pick 3 random test samples
indices = random.sample(range(len(test_dataset)), 3)

model_names = [cfg["name"] for cfg in MODEL_CONFIGS]
row_labels = ["Ground Truth", "FBP"] + model_names

fig, axes = plt.subplots(5, 3, figsize=(8, 14))
fig.subplots_adjust(wspace=-0.2, hspace=0.02)

with torch.no_grad():
    for col, idx in enumerate(indices):
        sino, img = test_dataset[idx]

        # Row 0: Ground truth
        axes[0, col].imshow(img.numpy().squeeze(), cmap="gray")

        # Row 1: FBP reconstruction
        observation, _ = test_pairs[idx]
        fbp_reco = np.asarray(fbp_reconstructor.reconstruct(observation))
        axes[1, col].imshow(fbp_reco, cmap="gray")

        # Rows 2-4: Neural network models
        sino_in = sino.unsqueeze(0).unsqueeze(0).to(device)
        for row, model in enumerate(models, start=2):
            output = model(sino_in)
            axes[row, col].imshow(output.squeeze().cpu().numpy(), cmap="gray")

# Hide ticks/spines, add label above each row via the middle column title
for row in range(5):
    axes[row, 1].set_title(row_labels[row], fontsize=12, fontweight="bold", pad=4)
    for col in range(3):
        axes[row, col].axis("off")

plt.show()